# Coding Pipeline for Bachelor's Thesis Project - Machine and Deep Learning Approaches to Heart Rate Estimation from Speech with Interpretability and Fairness Analysis

## Data loading and exploration

Start by verifying that the extracted feature dataset is internally consistent and ready for speaker-independent modeling. Since your project emphasizes speaker separation, fairness, and subgroup evaluation, this stage should check whether the rows, labels, and metadata all line up before any modeling begins.

Suggested exploration steps:

	•	Inspect table shape, feature types, missing values, duplicates, and target distribution.
	•	Check how many recordings each speaker contributes, and whether HR values are repeated across tasks or conditions.
	•	Plot the target distribution of heart rate, plus distributions by sex, language, age group, and task type.
	•	Examine correlations among features, especially among MFCCs, and look for constant or near-constant columns.
	•	Check whether any speakers or groups are underrepresented, because that will affect fairness analysis later.

Useful starter outputs:

	•	Summary statistics for all numeric features.
	•	Missingness heatmap.
	•	Histograms of HR and selected feature families.
	•	Boxplots of HR by demographic group.
	•	Scatter or pair plots for a small subset of important features.

In [ ]:
import pandas as pd

df = pd.read_pickle("dataset.pkl")
#metadata_df = pd.read_csv("tesdhe_metadata.csv")

df.head()

The dataset was also checked for any missing values
in the extracted MFCC coeﬃcients and was then normalized using MinMax normalizer to scale the MFCC
coeﬃcients in the range of [0,1] interval. Rows which
had missing MFCC coeﬃcients were discarded and not
used in the analysis. Of the 42 × 20= 840 rows of
MFCCcoeﬃcients,60rowswerediscardedduetomiss-
ing MFCC coeﬃcients. This results in a total of 780
rows of MFCC coeﬃcients with their corresponding
HR values. A histogram of HR values corresponding
to each of these MFCC coeﬃcients is shown in Fig. 1.
Normalization is achieved by shifting the values of each
MFCC coeﬃcient (denoted as x) so that the minimal
value is 0, and dividing by the new maximal coeﬃcient
value, as follows:
Normalized value=
x−min(x)
[max(x)−min(x)].

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- Config ----------
DATA_PATH = "features.csv"

TARGET_COL = "heart_rate_bpm"
SPEAKER_COL = "speaker_id"
SEX_COL = "sex"
LANG_COL = "language"
AGE_COL = "age"

OUTPUT_DIR = "outputs/exploration"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- Load ----------
df = pd.read_csv(DATA_PATH)

# ---------- Basic overview ----------
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDtypes:")
print(df.dtypes)

print("\nFirst rows:")
print(df.head())

print("\nBasic numeric summary:")
print(df.describe(include=[np.number]).T)

print("\nCategorical summary:")
for col in [SEX_COL, LANG_COL]:
    if col in df.columns:
        print(f"\nValue counts: {col}")
        print(df[col].value_counts(dropna=False))

# ---------- Speaker-level overview ----------
print("\nUnique speakers:", df[SPEAKER_COL].nunique())

samples_per_speaker = df.groupby(SPEAKER_COL).size().sort_values(ascending=False)
print("\nSamples per speaker:")
print(samples_per_speaker.describe())

samples_per_speaker.to_csv(f"{OUTPUT_DIR}/samples_per_speaker.csv")

# ---------- Missing values ----------
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("\nMissing values:")
print(missing)

missing.to_csv(f"{OUTPUT_DIR}/missing_values.csv", header=["n_missing"])

# ---------- Save full summary ----------
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_missing": df.isna().sum(),
    "n_unique": df.nunique(dropna=False)
})
summary.to_csv(f"{OUTPUT_DIR}/dataset_summary.csv")

# ---------- Plots ----------
sns.set(style="whitegrid")

# Target distribution
plt.figure(figsize=(8, 5))
sns.histplot(df[TARGET_COL], kde=True, bins=30)
plt.title("Heart Rate Distribution")
plt.xlabel("Heart Rate (BPM)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/heart_rate_distribution.png", dpi=300)
plt.close()

# Samples per speaker
plt.figure(figsize=(10, 5))
sns.histplot(samples_per_speaker, bins=30)
plt.title("Distribution of Samples per Speaker")
plt.xlabel("Number of samples")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/samples_per_speaker_distribution.png", dpi=300)
plt.close()

# Heart rate by sex
if SEX_COL in df.columns:
    plt.figure(figsize=(7, 5))
    sns.boxplot(data=df, x=SEX_COL, y=TARGET_COL)
    plt.title("Heart Rate by Sex")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/heart_rate_by_sex.png", dpi=300)
    plt.close()

# Heart rate by language
if LANG_COL in df.columns:
    plt.figure(figsize=(7, 5))
    sns.boxplot(data=df, x=LANG_COL, y=TARGET_COL)
    plt.title("Heart Rate by Language")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/heart_rate_by_language.png", dpi=300)
    plt.close()

# Age histogram
if AGE_COL in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[AGE_COL], bins=20, kde=True)
    plt.title("Age Distribution")
    plt.xlabel("Age")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/age_distribution.png", dpi=300)
    plt.close()

# Correlation heatmap for a subset of numeric features
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
subset_cols = [TARGET_COL] + [c for c in numeric_cols if c != TARGET_COL][:25]

corr = df[subset_cols].corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Subset of Numeric Features)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/correlation_heatmap_subset.png", dpi=300)
plt.close()


## Data quality and leakage checks

Before splitting the data, confirm that the dataset is suitable for regression and fair evaluation. Your proposal explicitly mentions preventing leakage by keeping speakers separate across train and test splits, so this check should happen before any feature scaling or selection.

What to verify:

	•	Speaker ID exists and is unique enough for group-wise splitting.
	•	No speaker appears in both train and test folds.
	•	No target leakage from features that directly encode HR or post-recording conditions in a way that would trivially reveal the label.
	•	Missing values are handled consistently, especially if some MFCC rows are incomplete.
	•	Outliers in HR and feature values are identified, not silently removed without justification.

A good practice is to create a small “data audit” notebook or script that saves:

	•	A data dictionary.
	•	A missing-value report.
	•	A group-count table.
	•	A leakage check log.

## Train-test design: Speaker-independent splitting
"Since the dataset has more MFCC frames than in-
dividuals samples, in this study, we have subjected the
data to (nT = 80%/nt = 20%) split to ensure more
samples are available for training and learning."

Your next coding pipeline should define the evaluation protocol before model building. Because your proposal uses speaker-independent cross-validation and subgroup metrics, the split strategy is a core part of the experiment rather than a later detail.

Recommended split logic:

	•	Hold out speakers, not rows.
	•	Use nested speaker-independent cross-validation if feasible.
	•	Keep the final test set untouched until the end.
	•	Stratify as much as possible by sex, language, or age group at the speaker level if the sample sizes allow it.

A practical design is:

	•	Outer loop: speaker-wise 10-fold CV for evaluation.
	•	Inner loop: speaker-wise validation split or grid search.
	•	Final holdout: one independent speaker set for the last comparison, if your sample size supports it.

# Model training

## Baseline model - mean predictor

Before training CNN-R and XGBoost, build simple baselines so you can tell whether the more complex models truly help. This is especially important for a thesis, because a strong baseline makes the results easier to defend.

For each baseline, record:

	•	RMSE.
	•	MAE.
	•	R2.
	•	Per-fold results, not only averages.
    
This gives you a reference point for judging whether XGBoost and CNN-R are actually improving prediction.

## Preprocessing pipeline

Once the split strategy is fixed, build a preprocessing pipeline that is fit only on training data. Your proposal already mentions MinMax normalization and possible row removal for missing MFCC values, so that logic should be encoded explicitly and reproducibly.

Recommended preprocessing steps:

	•	Remove duplicate rows and impossible values.
	•	Impute or drop missing features using a rule you justify.
	•	Normalize numeric features with training-only fit.
	•	Optionally remove highly correlated or near-zero-variance features.
	•	Keep a versioned feature list so experiments are reproducible.

If you keep all 385 MFCCs plus additional acoustic features, consider a modular feature selector so you can compare:

	•	All features.
	•	MFCC-only.
	•	Interpretable speech-feature subset.
	•	Selected features from importance ranking.

## XGBoost training and tuning

•	Tune depth, learning rate, number of estimators, subsample, and column sampling.
•	Use speaker-independent cross-validation.
•	Track feature importance and SHAP later.

## CNN-R training and tuning

•	Decide on input shape, likely fixed-length feature vectors or framed sequences.
•	Tune number of convolution layers, filters, kernel size, dropout, dense layers, and learning rate.
•	Use early stopping and model checkpointing.
•	Save the best fold model and the final best model separately.

### Coarse Random Search for Hyperparameter Tuning

In [ ]:
# optimizer&hypermarameter tuning notebook DL

### Fine Random Search for Hyperparameter Tuning

## Model comparison

After training, compare the models on exactly the same folds and the same evaluation protocol. Because your thesis aims to improve on prior work, the comparison should be statistically transparent rather than just descriptive.

Compare:

	•	Mean RMSE across folds.
	•	Mean MAE across folds.
	•	Mean  across folds.
	•	Fold-by-fold prediction errors.
	•	Error distributions by subgroup.

Also report:

	•	Whether the improvement over the baseline is consistent across folds.
	•	Which model is more stable.
	•	Whether one model performs better for certain groups.
    
A simple table with fold means and standard deviations is usually enough for the thesis chapter, but keep the raw fold outputs in CSV.

## Interpretability with SHAP

Once you have the best-performing model, add interpretability. Your proposal mentions SHAP and Pearson correlations, so this part should connect the predictive model to human-readable acoustic cues.

Recommended steps:

	•	Compute SHAP values for the best model.
	•	Rank features by average absolute SHAP value.
	•	Compare SHAP rankings to feature correlations with interpretable features like F0, jitter, shimmer, entropy, skewness, and kurtosis.
	•	Check whether the top features are stable across folds.
	•	Visualize a few individual predictions with local explanations.

Useful outputs:

	•	Global SHAP summary plot.
	•	Bar plot of top 10 features.
	•	Dependence plots for the strongest predictors.
	•	A table linking top MFCCs to interpretable acoustic patterns.

## Fairness analysis and bias mitigation

Fairness should not be a separate afterthought; it should be integrated into model evaluation once the main models are trained. Since your proposal plans subgroup analysis for male/female, Tamil/English, and young/old, you should treat each subgroup as a reporting axis.

Evaluate:

	•	RMSE by subgroup.
	•	MAE by subgroup.
	•	 by subgroup.
	•	Error gap between groups.
	•	Per-fold subgroup differences.

Then test whether differences are statistically meaningful:

	•	Paired t-tests on fold-wise errors.
	•	Confidence intervals for subgroup error gaps.
	•	If possible, bootstrap the error difference for robustness.

If unfairness appears:

	•	Reweight samples.
	•	Try Fairlearn-style regression constraints or bounded group loss.
	•	Compare before/after subgroup errors.

## Final reporting and figures

The final coding pipeline should end with a clean set of thesis-ready deliverables. This makes the analysis easier to write up and also protects you from last-minute confusion.

Produce:

	•	A final results CSV with all fold metrics.
	•	A subgroup metrics CSV.
	•	A feature-importance CSV.
	•	Figures for performance, SHAP, and fairness gaps.
	•	A reproducible notebook or script pipeline.
	•	A short experiment log listing every configuration you ran.